# 00 – Setup dan benchmark (CP0)

Notebook hanya memanggil fungsi di `src/`. Pekerjaan berat dijalankan sebagai skrip terpisah.

**Kejujuran hasil:** benchmark memakai tensor acak (cache data belum ada sampai CP2), jadi waktu epoch adalah ekstrapolasi, bukan epoch nyata. Bila tidak ada GPU, uji GPU dilaporkan `TIDAK_TERSEDIA`, bukan lulus. Benchmark dari mesin lain tidak dipakai ulang: berkas lama dipindah ke `*.bak`.

In [ ]:
import json, os, sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

def _root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'configs' / 'base.yaml').exists():
            return q
    raise FileNotFoundError('configs/base.yaml tidak ditemukan')
ROOT = _root(); os.chdir(ROOT); sys.path.insert(0, str(ROOT))

from src.config import load_merged, P
from src.launch import run_module

CFG = 'configs/base.yaml'
cfg = load_merged(CFG)
DEVICES = None      # mis. [0, 1]: satu rantai job per GPU lewat CUDA_VISIBLE_DEVICES (PRD §8)
REPS, FOLDS = None, None   # None = semua. Membatasi JOB YANG DILUNCURKAN saja, bukan memilih hasil; gerbang butuh semuanya.

import platform
OUT = P(cfg, 'cp0'); OUT.mkdir(parents=True, exist_ok=True)

def fresh(p):
    """True bila berkas ada DAN berasal dari mesin ini; berkas dari mesin lain dipindah ke .bak."""
    if not p.exists():
        return False
    host = json.loads(p.read_text()).get('host')
    if host == platform.node():
        return True
    p.rename(p.with_suffix('.json.bak'))
    return False
print('keluaran CP0 ->', OUT, '| host:', platform.node())

## 1. Uji lingkungan (impor, dataset baca-saja, bobot Swin-T, uji GPU)

In [ ]:
rc = run_module('src.env_check', ['--config', CFG, '--out', str(OUT / 'env_check.json')])
print('kode keluar:', rc)

## 2. Benchmark Swin-T (dilewati bila `benchmark.json` dari mesin ini sudah ada)

In [ ]:
rc = run_module('src.benchmark', ['--config', CFG, '--out', str(OUT / 'benchmark.json')],
                skip_if_exists=OUT / 'benchmark.json' if fresh(OUT / 'benchmark.json') else None)
print('kode keluar:', rc)

In [ ]:
bm = json.loads((OUT / 'benchmark.json').read_text())
for k in ['device', 'gpu', 'amp', 'data', 'batch_size', 'torch_threads', 'cpu_count', 'mean_step_seconds', 'images_per_second',
          'assumed_train_images', 'estimated_epoch_minutes', 'estimated_30_epoch_hours_per_job', 'peak_gpu_gb', 'ram_rss_gb']:
    print(f'{k}: {bm.get(k)}')
print('\nBatas atas S1 (10 lipatan x 30 epoch, tanpa henti dini):', round(bm['estimated_30_epoch_hours_per_job'] * 10, 1), 'jam / jumlah GPU')

## 3. Benchmark GAN (`src.bench_gan`): dasar menetapkan `gan.total_kimg` dan menghitung waktu total (PRD §8 Fase 0)

In [ ]:
rc = run_module('src.bench_gan', ['--config', 'configs/s6.yaml', '--out', str(OUT / 'benchmark_gan.json'), '--iters', '5', '--warmup', '2'],
                skip_if_exists=OUT / 'benchmark_gan.json' if fresh(OUT / 'benchmark_gan.json') else None)
print('kode keluar:', rc)

## 4. Pemeriksaan checkpoint CP0 dan `manifest.json` (sel terakhir)

In [ ]:
from src.checkpoint import finalize_cp0
report = finalize_cp0(OUT)
assert report['checkpoint'] == 'CP0'